# Import Data & Functions

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("https://raw.githubusercontent.com/sw1kwon/usa-elections/refs/heads/main/clean/Presidential_Elections2/2024-president_v1.csv")\
    .drop(columns=['county_fips', 'office', 'version'])
df.head()

,year,state,state_po,county_name,candidate,party,candidatevotes,totalvotes,mode
0,2024,ALABAMA,AL,AUTAUGA,CHASE OLIVER,LIBERTARIAN,65,28281,TOTAL VOTES
1,2024,ALABAMA,AL,AUTAUGA,DONALD J TRUMP,REPUBLICAN,20484,28281,TOTAL VOTES
2,2024,ALABAMA,AL,AUTAUGA,KAMALA D HARRIS,DEMOCRAT,7439,28281,TOTAL VOTES
3,2024,ALABAMA,AL,AUTAUGA,OTHER,OTHER,293,28281,TOTAL VOTES
4,2024,ALABAMA,AL,BALDWIN,CHASE OLIVER,LIBERTARIAN,241,122249,TOTAL VOTES


In [2]:
def process_state(state_name):
    """
    특정 state의 투표 데이터를 처리하여 집계 결과를 반환

    Parameters:
    state_name (str): 주 이름 (예: 'ALABAMA')

    Returns:
    pd.DataFrame: 집계된 투표 결과
    """
    # 해당 주 데이터 추출
    state_df = df.query(f"state == '{state_name}'")

    # party 열의 각 원소 개수 확인
    party_counts = state_df['party'].value_counts()
    print(f"\n{state_name} party counts:")
    print(party_counts)

    # 모든 party가 같은 개수인지 확인
    if party_counts.nunique() == 1:
        num_parties = len(party_counts)  # 고유한 party 개수

        # totalvotes 계산 (party 종류 개수로 나눔)
        total_votes = state_df['totalvotes'].sum() / num_parties

        # totalvotes가 정수인지 확인
        if total_votes != int(total_votes):
            print(f"Warning: totalvotes is not an integer: {total_votes}")

        # 각 party별 득표수 계산
        votes_republican = state_df.query("party == 'REPUBLICAN'")['candidatevotes'].sum()
        votes_democrat = state_df.query("party == 'DEMOCRAT'")['candidatevotes'].sum()
        votes_libertarian = state_df.query("party == 'LIBERTARIAN'")['candidatevotes'].sum()
        votes_other = state_df.query("party == 'OTHER'")['candidatevotes'].sum()

        # 결과 데이터프레임 생성
        result = pd.DataFrame({
            'year': [state_df['year'].iloc[0]],
            'state': [state_name],
            'state_po': [state_df['state_po'].iloc[0]],
            'totalvotes': [int(total_votes)],
            'votes_republican': [votes_republican],
            'votes_democrat': [votes_democrat],
            'votes_libertarian': [votes_libertarian],
            'votes_other': [votes_other]
        })

        return result
    else:
        print(f"Error: Party counts are not equal for {state_name}")
        return None

In [3]:
def process_state(state_name, check_equal_counts=True):
    """
    특정 state의 투표 데이터를 처리하여 집계 결과를 반환

    Parameters:
    state_name (str): 주 이름 (예: 'ALABAMA')
    check_equal_counts (bool): party별 개수가 같은지 확인할지 여부 (기본값: True)

    Returns:
    pd.DataFrame: 집계된 투표 결과
    """
    # 해당 주 데이터 추출
    state_df = df.query(f"state == '{state_name}'")

    # party 열의 각 원소 개수 확인
    party_counts = state_df['party'].value_counts()
    print(f"\n{state_name} party counts:")
    print(party_counts)

    # party 개수 확인 여부에 따라 처리
    if check_equal_counts and party_counts.nunique() != 1:
        print(f"Error: Party counts are not equal for {state_name}")
        return None

    # 데이터 처리
    num_parties = len(party_counts)  # 고유한 party 개수

    # totalvotes 계산 (party 종류 개수로 나눔)
    total_votes = state_df['totalvotes'].sum() / num_parties

    # totalvotes가 정수인지 확인
    if total_votes != int(total_votes):
        print(f"Warning: totalvotes is not an integer: {total_votes}")

    # 각 party별 득표수 계산
    votes_republican = state_df.query("party == 'REPUBLICAN'")['candidatevotes'].sum()
    votes_democrat = state_df.query("party == 'DEMOCRAT'")['candidatevotes'].sum()
    votes_libertarian = state_df.query("party == 'LIBERTARIAN'")['candidatevotes'].sum()
    votes_other = state_df.query("party == 'OTHER'")['candidatevotes'].sum()

    # 결과 데이터프레임 생성
    result = pd.DataFrame({
        'year': [state_df['year'].iloc[0]],
        'state': [state_name],
        'state_po': [state_df['state_po'].iloc[0]],
        'totalvotes': [int(total_votes)],
        'votes_republican': [votes_republican],
        'votes_democrat': [votes_democrat],
        'votes_libertarian': [votes_libertarian],
        'votes_other': [votes_other]
    })

    return result

In [4]:
def process_state2(state_name, check_equal_counts=True):
    """
    특정 state의 투표 데이터를 처리하여 집계 결과를 반환

    Parameters:
    state_name (str): 주 이름 (예: 'ALABAMA')
    check_equal_counts (bool): party별 개수가 같은지 확인할지 여부 (기본값: True)

    Returns:
    pd.DataFrame: 집계된 투표 결과
    """
    # 해당 주 데이터 추출
    state_df = df.query(f"state == '{state_name}'")

    # party 열의 각 원소 개수 확인
    party_counts = state_df['party'].value_counts()
    print(f"\n{state_name} party counts:")
    print(party_counts)

    # OTHER를 제외한 party들의 개수 확인
    non_other_counts = party_counts.drop('OTHER', errors='ignore')

    # party 개수 확인 여부에 따라 처리
    if check_equal_counts and non_other_counts.nunique() != 1:
        print(f"Error: Non-OTHER party counts are not equal for {state_name}")
        return None

    # county별로 OTHER 개수 확인
    other_counts_by_county = (state_df
                              .query("party == 'OTHER'")
                              .groupby('county_name')
                              .size())

    print(f"\nOTHER counts by county:")
    print(other_counts_by_county)

    # 각 county별로 총 party 개수 계산
    total_parties_by_county = state_df.groupby('county_name')['party'].nunique()

    print(f"\nTotal parties by county:")
    print(total_parties_by_county.value_counts())

    # totalvotes 계산: 각 county의 totalvotes를 한 번씩만 더함 (이미 총 투표수이므로)
    total_votes = state_df.groupby('county_name')['totalvotes'].first().sum()

    # totalvotes가 정수인지 확인
    if total_votes != int(total_votes):
        print(f"Warning: totalvotes is not an integer: {total_votes}")

    # 각 party별 득표수 계산
    votes_republican = state_df.query("party == 'REPUBLICAN'")['candidatevotes'].sum()
    votes_democrat = state_df.query("party == 'DEMOCRAT'")['candidatevotes'].sum()
    votes_libertarian = state_df.query("party == 'LIBERTARIAN'")['candidatevotes'].sum()
    votes_other = state_df.query("party == 'OTHER'")['candidatevotes'].sum()

    # 결과 데이터프레임 생성
    result = pd.DataFrame({
        'year': [state_df['year'].iloc[0]],
        'state': [state_name],
        'state_po': [state_df['state_po'].iloc[0]],
        'totalvotes': [int(total_votes)],
        'votes_republican': [votes_republican],
        'votes_democrat': [votes_democrat],
        'votes_libertarian': [votes_libertarian],
        'votes_other': [votes_other]
    })

    return result

In [5]:
def process_state3(state_name, check_equal_counts=True):
    """
    특정 state의 투표 데이터를 처리하여 집계 결과를 반환 (OVERVOTES, UNDERVOTES 제외)

    Parameters:
    state_name (str): 주 이름 (예: 'DISTRICT OF COLUMBIA')
    check_equal_counts (bool): party별 개수가 같은지 확인할지 여부 (기본값: True)

    Returns:
    pd.DataFrame: 집계된 투표 결과
    """
    # 해당 주 데이터 추출
    state_df = df.query(f"state == '{state_name}'")

    # OVERVOTES, UNDERVOTES 제외
    state_df = state_df[~state_df['candidate'].str.contains('OVERVOTES|UNDERVOTES', case=False, na=False)]

    # party 열의 각 원소 개수 확인
    party_counts = state_df['party'].value_counts()
    print(f"\n{state_name} party counts (excluding OVERVOTES/UNDERVOTES):")
    print(party_counts)

    # OTHER를 제외한 party들의 개수 확인
    non_other_counts = party_counts.drop('OTHER', errors='ignore')

    # party 개수 확인 여부에 따라 처리
    if check_equal_counts and non_other_counts.nunique() != 1:
        print(f"Error: Non-OTHER party counts are not equal for {state_name}")
        return None

    # county별로 OTHER 개수 확인
    other_counts_by_county = (state_df
                              .query("party == 'OTHER'")
                              .groupby('county_name')
                              .size())

    print(f"\nOTHER counts by county:")
    if len(other_counts_by_county) > 0:
        print(other_counts_by_county)
    else:
        print("No OTHER data")

    # 각 county별로 총 party 개수 계산
    total_parties_by_county = state_df.groupby('county_name')['party'].nunique()

    print(f"\nTotal parties by county:")
    print(total_parties_by_county.value_counts())

    # totalvotes 계산: 각 county의 totalvotes를 한 번씩만 더함
    total_votes = state_df.groupby('county_name')['totalvotes'].first().sum()

    # totalvotes가 정수인지 확인
    if total_votes != int(total_votes):
        print(f"Warning: totalvotes is not an integer: {total_votes}")

    # 각 party별 득표수 계산
    votes_republican = state_df.query("party == 'REPUBLICAN'")['candidatevotes'].sum()
    votes_democrat = state_df.query("party == 'DEMOCRAT'")['candidatevotes'].sum()
    votes_libertarian = state_df.query("party == 'LIBERTARIAN'")['candidatevotes'].sum()
    votes_other = state_df.query("party == 'OTHER'")['candidatevotes'].sum()

    # 결과 데이터프레임 생성
    result = pd.DataFrame({
        'year': [state_df['year'].iloc[0]],
        'state': [state_name],
        'state_po': [state_df['state_po'].iloc[0]],
        'totalvotes': [int(total_votes)],
        'votes_republican': [votes_republican],
        'votes_democrat': [votes_democrat],
        'votes_libertarian': [votes_libertarian],
        'votes_other': [votes_other]
    })

    return result

In [6]:
def process_state4(state_name, check_equal_counts=True):
    """
    특정 state의 투표 데이터를 처리하여 집계 결과를 반환 (LIBERTARIAN 일부 없는 경우)

    Parameters:
    state_name (str): 주 이름 (예: 'ILLINOIS')
    check_equal_counts (bool): party별 개수가 같은지 확인할지 여부 (기본값: True)

    Returns:
    pd.DataFrame: 집계된 투표 결과
    """
    # 해당 주 데이터 추출
    state_df = df.query(f"state == '{state_name}'")

    # party 열의 각 원소 개수 확인
    party_counts = state_df['party'].value_counts()
    print(f"\n{state_name} party counts:")
    print(party_counts)

    # LIBERTARIAN을 제외한 party들의 개수 확인
    non_lib_counts = party_counts.drop('LIBERTARIAN', errors='ignore')

    # party 개수 확인 여부에 따라 처리
    if check_equal_counts and non_lib_counts.nunique() != 1:
        print(f"Error: Non-LIBERTARIAN party counts are not equal for {state_name}")
        return None

    # LIBERTARIAN이 없는 county 찾기
    all_counties = set(state_df['county_name'].unique())
    libertarian_counties = set(state_df.query("party == 'LIBERTARIAN'")['county_name'].unique())
    counties_without_libertarian = all_counties - libertarian_counties

    print(f"\nCounties without LIBERTARIAN:")
    if len(counties_without_libertarian) > 0:
        print(sorted(counties_without_libertarian))
    else:
        print("All counties have LIBERTARIAN data")

    # 각 county별로 총 party 개수 계산
    total_parties_by_county = state_df.groupby('county_name')['party'].nunique()

    print(f"\nTotal parties by county:")
    print(total_parties_by_county.value_counts())

    # totalvotes 계산: 각 county의 totalvotes를 한 번씩만 더함
    total_votes = state_df.groupby('county_name')['totalvotes'].first().sum()

    # totalvotes가 정수인지 확인
    if total_votes != int(total_votes):
        print(f"Warning: totalvotes is not an integer: {total_votes}")

    # 각 party별 득표수 계산
    votes_republican = state_df.query("party == 'REPUBLICAN'")['candidatevotes'].sum()
    votes_democrat = state_df.query("party == 'DEMOCRAT'")['candidatevotes'].sum()
    votes_libertarian = state_df.query("party == 'LIBERTARIAN'")['candidatevotes'].sum()
    votes_other = state_df.query("party == 'OTHER'")['candidatevotes'].sum()

    # 결과 데이터프레임 생성
    result = pd.DataFrame({
        'year': [state_df['year'].iloc[0]],
        'state': [state_name],
        'state_po': [state_df['state_po'].iloc[0]],
        'totalvotes': [int(total_votes)],
        'votes_republican': [votes_republican],
        'votes_democrat': [votes_democrat],
        'votes_libertarian': [votes_libertarian],
        'votes_other': [votes_other]
    })

    return result

In [7]:
def process_state5(state_name, check_equal_counts=True):
    """
    특정 state의 투표 데이터를 처리하여 집계 결과를 반환 (OVERVOTES, UNDERVOTES 제외)

    Parameters:
    state_name (str): 주 이름 (예: 'ARIZONA')
    check_equal_counts (bool): party별 개수가 같은지 확인할지 여부 (기본값: True)

    Returns:
    pd.DataFrame: 집계된 투표 결과
    """
    # 해당 주 데이터 추출 후 mode가 'TOTAL VOTES'인 행만 필터링
    state_df = df.query(f"state == '{state_name}' and mode == 'TOTAL VOTES'")

    # OVERVOTES, UNDERVOTES 제외
    state_df = state_df[~state_df['candidate'].str.contains('OVERVOTES|UNDERVOTES', case=False, na=False)]

    # party 열의 각 원소 개수 확인
    party_counts = state_df['party'].value_counts()
    print(f"\n{state_name} party counts:")
    print(party_counts)

    # party 개수 확인 여부에 따라 처리
    if check_equal_counts and party_counts.nunique() != 1:
        print(f"Error: Party counts are not equal for {state_name}")
        return None

    # totalvotes 계산: candidatevotes 열의 모든 합
    total_votes = state_df['candidatevotes'].sum()

    # totalvotes가 정수인지 확인
    if total_votes != int(total_votes):
        print(f"Warning: totalvotes is not an integer: {total_votes}")

    # 각 party별 득표수 계산
    votes_republican = state_df.query("party == 'REPUBLICAN'")['candidatevotes'].sum()
    votes_democrat = state_df.query("party == 'DEMOCRAT'")['candidatevotes'].sum()
    votes_libertarian = state_df.query("party == 'LIBERTARIAN'")['candidatevotes'].sum()
    votes_other = state_df.query("party == 'OTHER'")['candidatevotes'].sum()

    # 결과 데이터프레임 생성
    result = pd.DataFrame({
        'year': [state_df['year'].iloc[0]],
        'state': [state_name],
        'state_po': [state_df['state_po'].iloc[0]],
        'totalvotes': [int(total_votes)],
        'votes_republican': [votes_republican],
        'votes_democrat': [votes_democrat],
        'votes_libertarian': [votes_libertarian],
        'votes_other': [votes_other]
    })

    return result

In [8]:
def process_state6(state_name, check_equal_counts=True):
    """
    특정 state의 투표 데이터를 처리하여 집계 결과를 반환

    Parameters:
    state_name (str): 주 이름 (예: 'ARKANSAS')
    check_equal_counts (bool): party별 개수가 같은지 확인할지 여부 (기본값: True)

    Returns:
    pd.DataFrame: 집계된 투표 결과
    """
    # 해당 주 데이터 추출 후 mode가 'TOTAL VOTES'인 행만 필터링
    state_df = df.query(f"state == '{state_name}' and mode == 'TOTAL VOTES'")

    # party 열의 각 원소 개수 확인
    party_counts = state_df['party'].value_counts()
    print(f"\n{state_name} party counts:")
    print(party_counts)

    # party 개수 확인 여부에 따라 처리
    if check_equal_counts and party_counts.nunique() != 1:
        print(f"Error: Party counts are not equal for {state_name}")
        return None

    # totalvotes 계산: candidatevotes 열의 모든 합
    total_votes = state_df['candidatevotes'].sum()

    # totalvotes가 정수인지 확인
    if total_votes != int(total_votes):
        print(f"Warning: totalvotes is not an integer: {total_votes}")

    # 각 party별 득표수 계산
    votes_republican = state_df.query("party == 'REPUBLICAN'")['candidatevotes'].sum()
    votes_democrat = state_df.query("party == 'DEMOCRAT'")['candidatevotes'].sum()
    votes_libertarian = state_df.query("party == 'LIBERTARIAN'")['candidatevotes'].sum()
    votes_other = state_df.query("party == 'OTHER'")['candidatevotes'].sum()

    # 결과 데이터프레임 생성
    result = pd.DataFrame({
        'year': [state_df['year'].iloc[0]],
        'state': [state_name],
        'state_po': [state_df['state_po'].iloc[0]],
        'totalvotes': [int(total_votes)],
        'votes_republican': [votes_republican],
        'votes_democrat': [votes_democrat],
        'votes_libertarian': [votes_libertarian],
        'votes_other': [votes_other]
    })

    return result

In [9]:
def process_state7(state_name, check_equal_counts=True):
    """
    특정 state의 투표 데이터를 처리하여 집계 결과를 반환 (TOTAL VOTES CAST 제외)

    Parameters:
    state_name (str): 주 이름 (예: 'SOUTH CAROLINA')
    check_equal_counts (bool): party별 개수가 같은지 확인할지 여부 (기본값: True)

    Returns:
    pd.DataFrame: 집계된 투표 결과
    """
    # 해당 주 데이터 추출 후 mode가 'TOTAL VOTES'인 행만 필터링
    state_df = df.query(f"state == '{state_name}' and mode == 'TOTAL VOTES'")

    # TOTAL VOTES CAST 제외
    state_df = state_df[~state_df['candidate'].str.contains('TOTAL VOTES CAST', case=False, na=False)]

    # party 열의 각 원소 개수 확인
    party_counts = state_df['party'].value_counts()
    print(f"\n{state_name} party counts:")
    print(party_counts)

    # party 개수 확인 여부에 따라 처리
    if check_equal_counts and party_counts.nunique() != 1:
        print(f"Error: Party counts are not equal for {state_name}")
        return None

    # totalvotes 계산: candidatevotes 열의 모든 합
    total_votes = state_df['candidatevotes'].sum()

    # totalvotes가 정수인지 확인
    if total_votes != int(total_votes):
        print(f"Warning: totalvotes is not an integer: {total_votes}")

    # 각 party별 득표수 계산
    votes_republican = state_df.query("party == 'REPUBLICAN'")['candidatevotes'].sum()
    votes_democrat = state_df.query("party == 'DEMOCRAT'")['candidatevotes'].sum()
    votes_libertarian = state_df.query("party == 'LIBERTARIAN'")['candidatevotes'].sum()
    votes_other = state_df.query("party == 'OTHER'")['candidatevotes'].sum()

    # 결과 데이터프레임 생성
    result = pd.DataFrame({
        'year': [state_df['year'].iloc[0]],
        'state': [state_name],
        'state_po': [state_df['state_po'].iloc[0]],
        'totalvotes': [int(total_votes)],
        'votes_republican': [votes_republican],
        'votes_democrat': [votes_democrat],
        'votes_libertarian': [votes_libertarian],
        'votes_other': [votes_other]
    })

    return result

In [10]:
def process_state8(state_name, check_equal_counts=True):
    """
    특정 state의 투표 데이터를 처리하여 집계 결과를 반환

    Parameters:
    state_name (str): 주 이름 (예: 'SOUTH DAKOTA')
    check_equal_counts (bool): party별 개수가 같은지 확인할지 여부 (기본값: True)

    Returns:
    pd.DataFrame: 집계된 투표 결과
    """
    # 해당 주 데이터 추출 후 mode가 'TOTAL VOTES' 또는 'VOTE CENTER'인 행만 필터링
    state_df = df.query(f"state == '{state_name}' and (mode == 'TOTAL VOTES' or mode == 'VOTE CENTER')")

    # party 열의 각 원소 개수 확인
    party_counts = state_df['party'].value_counts()
    print(f"\n{state_name} party counts:")
    print(party_counts)

    # party 개수 확인 여부에 따라 처리
    if check_equal_counts and party_counts.nunique() != 1:
        print(f"Error: Party counts are not equal for {state_name}")
        return None

    # totalvotes 계산: candidatevotes 열의 모든 합
    total_votes = state_df['candidatevotes'].sum()

    # totalvotes가 정수인지 확인
    if total_votes != int(total_votes):
        print(f"Warning: totalvotes is not an integer: {total_votes}")

    # 각 party별 득표수 계산
    votes_republican = state_df.query("party == 'REPUBLICAN'")['candidatevotes'].sum()
    votes_democrat = state_df.query("party == 'DEMOCRAT'")['candidatevotes'].sum()
    votes_libertarian = state_df.query("party == 'LIBERTARIAN'")['candidatevotes'].sum()
    votes_other = state_df.query("party == 'OTHER'")['candidatevotes'].sum()

    # 결과 데이터프레임 생성
    result = pd.DataFrame({
        'year': [state_df['year'].iloc[0]],
        'state': [state_name],
        'state_po': [state_df['state_po'].iloc[0]],
        'totalvotes': [int(total_votes)],
        'votes_republican': [votes_republican],
        'votes_democrat': [votes_democrat],
        'votes_libertarian': [votes_libertarian],
        'votes_other': [votes_other]
    })

    return result

In [11]:
def process_state9(state_name):
    """
    특정 state의 투표 데이터를 처리하여 집계 결과를 반환

    Parameters:
    state_name (str): 주 이름 (예: 'NORTH CAROLINA')

    Returns:
    pd.DataFrame: 집계된 투표 결과
    """
    # 해당 주 데이터 추출
    state_df = df.query(f"state == '{state_name}'")

    # party 열의 각 원소 개수 확인
    party_counts = state_df['party'].value_counts()
    print(f"\n{state_name} party counts:")
    print(party_counts)

    # county별 행의 개수 확인
    county_row_counts = state_df.groupby('county_name').size()
    print(f"\nRow counts by county:")
    print(county_row_counts.value_counts())

    # totalvotes 계산: 각 county의 totalvotes 합 / 각 county의 행 개수로 나눈 값들의 합
    total_votes = 0
    for county in state_df['county_name'].unique():
        county_data = state_df.query(f"county_name == '{county}'")
        county_totalvotes = county_data['totalvotes'].sum()
        county_rows = len(county_data)
        total_votes += county_totalvotes / county_rows

    # totalvotes가 정수인지 확인
    if total_votes != int(total_votes):
        print(f"Warning: totalvotes is not an integer: {total_votes}")

    # 각 party별 득표수 계산
    votes_republican = state_df.query("party == 'REPUBLICAN'")['candidatevotes'].sum()
    votes_democrat = state_df.query("party == 'DEMOCRAT'")['candidatevotes'].sum()
    votes_libertarian = state_df.query("party == 'LIBERTARIAN'")['candidatevotes'].sum()
    votes_other = state_df.query("party == 'OTHER'")['candidatevotes'].sum()

    # 결과 데이터프레임 생성
    result = pd.DataFrame({
        'year': [state_df['year'].iloc[0]],
        'state': [state_name],
        'state_po': [state_df['state_po'].iloc[0]],
        'totalvotes': [int(total_votes)],
        'votes_republican': [votes_republican],
        'votes_democrat': [votes_democrat],
        'votes_libertarian': [votes_libertarian],
        'votes_other': [votes_other]
    })

    return result

In [12]:
def process_state10(state_name, check_equal_counts=True):
    """
    특정 state의 투표 데이터를 처리하여 집계 결과를 반환 (TOTAL VOTES CAST, UNDERVOTES 제외)

    Parameters:
    state_name (str): 주 이름 (예: 'VERMONT')
    check_equal_counts (bool): party별 개수가 같은지 확인할지 여부 (기본값: True)

    Returns:
    pd.DataFrame: 집계된 투표 결과
    """
    # 해당 주 데이터 추출
    state_df = df.query(f"state == '{state_name}'")

    # TOTAL VOTES CAST 또는 UNDERVOTES 제외
    state_df = state_df[~state_df['candidate'].str.contains('TOTAL VOTES CAST|UNDERVOTES', case=False, na=False)]

    # party 열의 각 원소 개수 확인
    party_counts = state_df['party'].value_counts()
    print(f"\n{state_name} party counts:")
    print(party_counts)

    # party 개수 확인 여부에 따라 처리
    if check_equal_counts and party_counts.nunique() != 1:
        print(f"Error: Party counts are not equal for {state_name}")
        return None

    # totalvotes 계산: candidatevotes 열의 모든 합
    total_votes = state_df['candidatevotes'].sum()

    # totalvotes가 정수인지 확인
    if total_votes != int(total_votes):
        print(f"Warning: totalvotes is not an integer: {total_votes}")

    # 각 party별 득표수 계산
    votes_republican = state_df.query("party == 'REPUBLICAN'")['candidatevotes'].sum()
    votes_democrat = state_df.query("party == 'DEMOCRAT'")['candidatevotes'].sum()
    votes_libertarian = state_df.query("party == 'LIBERTARIAN'")['candidatevotes'].sum()
    votes_other = state_df.query("party == 'OTHER'")['candidatevotes'].sum()

    # 결과 데이터프레임 생성
    result = pd.DataFrame({
        'year': [state_df['year'].iloc[0]],
        'state': [state_name],
        'state_po': [state_df['state_po'].iloc[0]],
        'totalvotes': [int(total_votes)],
        'votes_republican': [votes_republican],
        'votes_democrat': [votes_democrat],
        'votes_libertarian': [votes_libertarian],
        'votes_other': [votes_other]
    })

    return result

In [13]:
def process_state11(state_name, check_equal_counts=True):
    """
    특정 state의 투표 데이터를 처리하여 집계 결과를 반환 (TOTAL VOTES CAST 제외)

    Parameters:
    state_name (str): 주 이름 (예: 'WEST VIRGINIA')
    check_equal_counts (bool): party별 개수가 같은지 확인할지 여부 (기본값: True)

    Returns:
    pd.DataFrame: 집계된 투표 결과
    """
    # 해당 주 데이터 추출
    state_df = df.query(f"state == '{state_name}'")

    # TOTAL VOTES CAST 제외
    state_df = state_df[~state_df['candidate'].str.contains('TOTAL VOTES CAST', case=False, na=False)]

    # party 열의 각 원소 개수 확인
    party_counts = state_df['party'].value_counts()
    print(f"\n{state_name} party counts:")
    print(party_counts)

    # party 개수 확인 여부에 따라 처리
    if check_equal_counts and party_counts.nunique() != 1:
        print(f"Error: Party counts are not equal for {state_name}")
        return None

    # totalvotes 계산: candidatevotes 열의 모든 합
    total_votes = state_df['candidatevotes'].sum()

    # totalvotes가 정수인지 확인
    if total_votes != int(total_votes):
        print(f"Warning: totalvotes is not an integer: {total_votes}")

    # 각 party별 득표수 계산
    votes_republican = state_df.query("party == 'REPUBLICAN'")['candidatevotes'].sum()
    votes_democrat = state_df.query("party == 'DEMOCRAT'")['candidatevotes'].sum()
    votes_libertarian = state_df.query("party == 'LIBERTARIAN'")['candidatevotes'].sum()
    votes_other = state_df.query("party == 'OTHER'")['candidatevotes'].sum()

    # 결과 데이터프레임 생성
    result = pd.DataFrame({
        'year': [state_df['year'].iloc[0]],
        'state': [state_name],
        'state_po': [state_df['state_po'].iloc[0]],
        'totalvotes': [int(total_votes)],
        'votes_republican': [votes_republican],
        'votes_democrat': [votes_democrat],
        'votes_libertarian': [votes_libertarian],
        'votes_other': [votes_other]
    })

    return result

In [14]:
def process_state12(state_name, check_equal_counts=True):
    """
    특정 state의 투표 데이터를 처리하여 집계 결과를 반환 (OVERVOTES, UNDERVOTES 제외)

    Parameters:
    state_name (str): 주 이름 (예: 'WYOMING')
    check_equal_counts (bool): party별 개수가 같은지 확인할지 여부 (기본값: True)

    Returns:
    pd.DataFrame: 집계된 투표 결과
    """
    # 해당 주 데이터 추출
    state_df = df.query(f"state == '{state_name}'")

    # OVERVOTES 또는 UNDERVOTES 제외
    state_df = state_df[~state_df['candidate'].str.contains('OVERVOTES|UNDERVOTES', case=False, na=False)]

    # party 열의 각 원소 개수 확인
    party_counts = state_df['party'].value_counts()
    print(f"\n{state_name} party counts:")
    print(party_counts)

    # party 개수 확인 여부에 따라 처리
    if check_equal_counts and party_counts.nunique() != 1:
        print(f"Error: Party counts are not equal for {state_name}")
        return None

    # totalvotes 계산: candidatevotes 열의 모든 합
    total_votes = state_df['candidatevotes'].sum()

    # totalvotes가 정수인지 확인
    if total_votes != int(total_votes):
        print(f"Warning: totalvotes is not an integer: {total_votes}")

    # 각 party별 득표수 계산
    votes_republican = state_df.query("party == 'REPUBLICAN'")['candidatevotes'].sum()
    votes_democrat = state_df.query("party == 'DEMOCRAT'")['candidatevotes'].sum()
    votes_libertarian = state_df.query("party == 'LIBERTARIAN'")['candidatevotes'].sum()
    votes_other = state_df.query("party == 'OTHER'")['candidatevotes'].sum()

    # 결과 데이터프레임 생성
    result = pd.DataFrame({
        'year': [state_df['year'].iloc[0]],
        'state': [state_name],
        'state_po': [state_df['state_po'].iloc[0]],
        'totalvotes': [int(total_votes)],
        'votes_republican': [votes_republican],
        'votes_democrat': [votes_democrat],
        'votes_libertarian': [votes_libertarian],
        'votes_other': [votes_other]
    })

    return result

# TOTAL VOTES only

## AL, AK

In [15]:
df_AL = process_state('ALABAMA')
df_AL


ALABAMA party counts:
party
LIBERTARIAN    67
REPUBLICAN     67
DEMOCRAT       67
OTHER          67
Name: count, dtype: int64


,year,state,state_po,totalvotes,votes_republican,votes_democrat,votes_libertarian,votes_other
0,2024,ALABAMA,AL,2264972,1462616,772412,4930,25014


In [16]:
df_AK = process_state('ALASKA')
df_AK


ALASKA party counts:
party
LIBERTARIAN    41
REPUBLICAN     41
DEMOCRAT       41
OTHER          41
Name: count, dtype: int64


,year,state,state_po,totalvotes,votes_republican,votes_democrat,votes_libertarian,votes_other
0,2024,ALASKA,AK,338177,184458,140026,3040,10653


## CA, CO, CT

In [17]:
df_CA = process_state('CALIFORNIA', check_equal_counts=False)
df_CA # OTHER +


CALIFORNIA party counts:
party
OTHER          66
LIBERTARIAN    58
REPUBLICAN     58
DEMOCRAT       58
Name: count, dtype: int64


,year,state,state_po,totalvotes,votes_republican,votes_democrat,votes_libertarian,votes_other
0,2024,CALIFORNIA,CA,16049890,6081697,9276179,66662,438140


In [18]:
df_CA = process_state2('CALIFORNIA', check_equal_counts=False)
df_CA # OTHER +


CALIFORNIA party counts:
party
OTHER          66
LIBERTARIAN    58
REPUBLICAN     58
DEMOCRAT       58
Name: count, dtype: int64

OTHER counts by county:
county_name
ALAMEDA            1
ALPINE             1
AMADOR             1
BUTTE              1
CALAVERAS          1
COLUSA             1
CONTRA COSTA       1
DEL NORTE          1
EL DORADO          1
FRESNO             1
GLENN              1
HUMBOLDT           1
IMPERIAL           1
INYO               1
KERN               1
KINGS              1
LAKE               1
LASSEN             1
LOS ANGELES        1
MADERA             1
MARIN              1
MARIPOSA           1
MENDOCINO          1
MERCED             1
MODOC              1
MONO               1
MONTEREY           1
NAPA               1
NEVADA             1
ORANGE             1
PLACER             1
PLUMAS             1
RIVERSIDE          1
SACRAMENTO         1
SAN BENITO         1
SAN BERNARDINO     1
SAN DIEGO          1
SAN FRANCISCO      1
SAN JOAQUIN        1
SAN LUIS OBISP

,year,state,state_po,totalvotes,votes_republican,votes_democrat,votes_libertarian,votes_other
0,2024,CALIFORNIA,CA,15862678,6081697,9276179,66662,438140


In [19]:
df_CO = process_state('COLORADO')
df_CO


COLORADO party counts:
party
LIBERTARIAN    64
REPUBLICAN     64
DEMOCRAT       64
OTHER          64
Name: count, dtype: int64


,year,state,state_po,totalvotes,votes_republican,votes_democrat,votes_libertarian,votes_other
0,2024,COLORADO,CO,3190873,1377441,1728159,21439,63834


In [20]:
df_CT = process_state('CONNECTICUT', check_equal_counts=False)
df_CT # OTHER +


CONNECTICUT party counts:
party
OTHER          15
LIBERTARIAN     8
REPUBLICAN      8
DEMOCRAT        8
Name: count, dtype: int64


,year,state,state_po,totalvotes,votes_republican,votes_democrat,votes_libertarian,votes_other
0,2024,CONNECTICUT,CT,2085686,736918,992053,6729,23310


In [21]:
df_CT = process_state2('CONNECTICUT', check_equal_counts=False)
df_CT # OTHER +


CONNECTICUT party counts:
party
OTHER          15
LIBERTARIAN     8
REPUBLICAN      8
DEMOCRAT        8
Name: count, dtype: int64

OTHER counts by county:
county_name
FAIRFIELD     1
HARTFORD      2
LITCHFIELD    2
MIDDLESEX     2
NEW HAVEN     2
NEW LONDON    2
TOLLAND       2
WINDHAM       2
dtype: int64

Total parties by county:
party
4    8
Name: count, dtype: int64


,year,state,state_po,totalvotes,votes_republican,votes_democrat,votes_libertarian,votes_other
0,2024,CONNECTICUT,CT,1759010,736918,992053,6729,23310


## DE, DC, FL, GA

In [22]:
df_DE = process_state('DELAWARE')
df_DE


DELAWARE party counts:
party
LIBERTARIAN    3
REPUBLICAN     3
DEMOCRAT       3
OTHER          3
Name: count, dtype: int64


,year,state,state_po,totalvotes,votes_republican,votes_democrat,votes_libertarian,votes_other
0,2024,DELAWARE,DE,511697,214351,289758,2038,5550


In [23]:
df_DC = process_state('DISTRICT OF COLUMBIA', check_equal_counts=False)
df_DC # OVERVOTES UNDERVOTES


DISTRICT OF COLUMBIA party counts:
party
OTHER         3
REPUBLICAN    1
DEMOCRAT      1
Name: count, dtype: int64


,year,state,state_po,totalvotes,votes_republican,votes_democrat,votes_libertarian,votes_other
0,2024,DISTRICT OF COLUMBIA,DC,543131,21076,294185,0,13153


In [24]:
df_DC = process_state3('DISTRICT OF COLUMBIA', check_equal_counts=False)
df_DC # OVERVOTES UNDERVOTES


DISTRICT OF COLUMBIA party counts (excluding OVERVOTES/UNDERVOTES):
party
REPUBLICAN    1
DEMOCRAT      1
OTHER         1
Name: count, dtype: int64

OTHER counts by county:
county_name
DISTRICT OF COLUMBIA    1
dtype: int64

Total parties by county:
party
3    1
Name: count, dtype: int64


,year,state,state_po,totalvotes,votes_republican,votes_democrat,votes_libertarian,votes_other
0,2024,DISTRICT OF COLUMBIA,DC,325879,21076,294185,0,10618


In [25]:
df_FL = process_state('FLORIDA')
df_FL


FLORIDA party counts:
party
LIBERTARIAN    67
REPUBLICAN     67
DEMOCRAT       67
OTHER          67
Name: count, dtype: int64


,year,state,state_po,totalvotes,votes_republican,votes_democrat,votes_libertarian,votes_other
0,2024,FLORIDA,FL,10893752,6110125,4683038,31972,68617


In [26]:
df_GA = process_state('GEORGIA')
df_GA


GEORGIA party counts:
party
LIBERTARIAN    159
REPUBLICAN     159
DEMOCRAT       159
OTHER          159
Name: count, dtype: int64


,year,state,state_po,totalvotes,votes_republican,votes_democrat,votes_libertarian,votes_other
0,2024,GEORGIA,GA,5250047,2663117,2548017,20684,18229


## HI, IL, IN, KY

In [27]:
df_HI = process_state('HAWAII')
df_HI


HAWAII party counts:
party
LIBERTARIAN    4
REPUBLICAN     4
DEMOCRAT       4
OTHER          4
Name: count, dtype: int64


,year,state,state_po,totalvotes,votes_republican,votes_democrat,votes_libertarian,votes_other
0,2024,HAWAII,HI,516701,193661,313044,2733,7263


In [28]:
df_IL = process_state('ILLINOIS', check_equal_counts=False)
df_IL # 일부 선거구에 party에 LIBERTARIAN이 없음


ILLINOIS party counts:
party
REPUBLICAN     102
DEMOCRAT       102
OTHER          102
LIBERTARIAN     96
Name: count, dtype: int64


,year,state,state_po,totalvotes,votes_republican,votes_democrat,votes_libertarian,votes_other
0,2024,ILLINOIS,IL,5627460,2449079,3062863,3510,117858


In [29]:
df_IL = process_state4('ILLINOIS', check_equal_counts=False)
df_IL # 일부 선거구에 party에 LIBERTARIAN이 없음


ILLINOIS party counts:
party
REPUBLICAN     102
DEMOCRAT       102
OTHER          102
LIBERTARIAN     96
Name: count, dtype: int64

Counties without LIBERTARIAN:
['ALEXANDER', 'CALHOUN', 'HAMILTON', 'MARSHALL', 'STARK', 'WABASH']

Total parties by county:
party
4    96
3     6
Name: count, dtype: int64


,year,state,state_po,totalvotes,votes_republican,votes_democrat,votes_libertarian,votes_other
0,2024,ILLINOIS,IL,5633310,2449079,3062863,3510,117858


In [30]:
df_IN = process_state('INDIANA')
df_IN


INDIANA party counts:
party
LIBERTARIAN    92
REPUBLICAN     92
DEMOCRAT       92
OTHER          92
Name: count, dtype: int64


,year,state,state_po,totalvotes,votes_republican,votes_democrat,votes_libertarian,votes_other
0,2024,INDIANA,IN,2936677,1720347,1163603,20425,32302


In [31]:
df_KY = process_state('KENTUCKY')
df_KY


KENTUCKY party counts:
party
LIBERTARIAN    120
REPUBLICAN     120
DEMOCRAT       120
OTHER          120
Name: count, dtype: int64


,year,state,state_po,totalvotes,votes_republican,votes_democrat,votes_libertarian,votes_other
0,2024,KENTUCKY,KY,2074530,1337494,704043,6422,26571


## ME, MD, MA, MI

In [32]:
df_ME = process_state('MAINE')
df_ME


MAINE party counts:
party
LIBERTARIAN    16
REPUBLICAN     16
DEMOCRAT       16
OTHER          16
Name: count, dtype: int64


,year,state,state_po,totalvotes,votes_republican,votes_democrat,votes_libertarian,votes_other
0,2024,MAINE,ME,824806,376991,430342,5253,12220


In [33]:
df_MD = process_state('MARYLAND')
df_MD


MARYLAND party counts:
party
LIBERTARIAN    24
REPUBLICAN     24
DEMOCRAT       24
OTHER          24
Name: count, dtype: int64


,year,state,state_po,totalvotes,votes_republican,votes_democrat,votes_libertarian,votes_other
0,2024,MARYLAND,MD,3038334,1035550,1902577,15570,84637


In [34]:
df_MA = process_state('MASSACHUSETTS')
df_MA


MASSACHUSETTS party counts:
party
LIBERTARIAN    14
REPUBLICAN     14
DEMOCRAT       14
OTHER          14
Name: count, dtype: int64


,year,state,state_po,totalvotes,votes_republican,votes_democrat,votes_libertarian,votes_other
0,2024,MASSACHUSETTS,MA,3473668,1251303,2126518,17735,78112


In [35]:
df_MI = process_state('MICHIGAN')
df_MI


MICHIGAN party counts:
party
LIBERTARIAN    83
REPUBLICAN     83
DEMOCRAT       83
OTHER          83
Name: count, dtype: int64


,year,state,state_po,totalvotes,votes_republican,votes_democrat,votes_libertarian,votes_other
0,2024,MICHIGAN,MI,5664186,2816636,2736533,22440,88577


## MN, MS, MO, MT

In [36]:
df_MN = process_state('MINNESOTA')
df_MN


MINNESOTA party counts:
party
LIBERTARIAN    87
REPUBLICAN     87
DEMOCRAT       87
OTHER          87
Name: count, dtype: int64


,year,state,state_po,totalvotes,votes_republican,votes_democrat,votes_libertarian,votes_other
0,2024,MINNESOTA,MN,3253920,1519032,1656979,15155,62754


In [37]:
df_MS = process_state('MISSISSIPPI')
df_MS


MISSISSIPPI party counts:
party
LIBERTARIAN    82
REPUBLICAN     82
DEMOCRAT       82
OTHER          82
Name: count, dtype: int64


,year,state,state_po,totalvotes,votes_republican,votes_democrat,votes_libertarian,votes_other
0,2024,MISSISSIPPI,MS,1228008,747744,466668,2536,11060


In [38]:
df_MO = process_state('MISSOURI')
df_MO


MISSOURI party counts:
party
LIBERTARIAN    116
REPUBLICAN     116
DEMOCRAT       116
OTHER          116
Name: count, dtype: int64


,year,state,state_po,totalvotes,votes_republican,votes_democrat,votes_libertarian,votes_other
0,2024,MISSOURI,MO,2995327,1751986,1200599,23876,18866


In [39]:
df_MT = process_state('MONTANA')
df_MT


MONTANA party counts:
party
LIBERTARIAN    56
REPUBLICAN     56
DEMOCRAT       56
OTHER          56
Name: count, dtype: int64


,year,state,state_po,totalvotes,votes_republican,votes_democrat,votes_libertarian,votes_other
0,2024,MONTANA,MT,602963,352079,231906,4275,14703


## NE, NV, NH

In [40]:
df_NE = process_state('NEBRASKA')
df_NE


NEBRASKA party counts:
party
LIBERTARIAN    93
REPUBLICAN     93
DEMOCRAT       93
OTHER          93
Name: count, dtype: int64


,year,state,state_po,totalvotes,votes_republican,votes_democrat,votes_libertarian,votes_other
0,2024,NEBRASKA,NE,947159,564816,369995,6399,5949


In [41]:
df_NV = process_state('NEVADA')
df_NV


NEVADA party counts:
party
LIBERTARIAN    17
REPUBLICAN     17
DEMOCRAT       17
OTHER          17
Name: count, dtype: int64


,year,state,state_po,totalvotes,votes_republican,votes_democrat,votes_libertarian,votes_other
0,2024,NEVADA,NV,1484840,751205,705197,6059,22379


In [42]:
df_NH = process_state('NEW HAMPSHIRE')
df_NH


NEW HAMPSHIRE party counts:
party
LIBERTARIAN    10
REPUBLICAN     10
DEMOCRAT       10
OTHER          10
Name: count, dtype: int64


,year,state,state_po,totalvotes,votes_republican,votes_democrat,votes_libertarian,votes_other
0,2024,NEW HAMPSHIRE,NH,826189,395523,418488,4425,7753


## NJ, NY, ND

In [43]:
df_NJ = process_state('NEW JERSEY')
df_NJ


NEW JERSEY party counts:
party
LIBERTARIAN    21
REPUBLICAN     21
DEMOCRAT       21
OTHER          21
Name: count, dtype: int64


,year,state,state_po,totalvotes,votes_republican,votes_democrat,votes_libertarian,votes_other
0,2024,NEW JERSEY,NJ,4272725,1968215,2220713,10500,73297


In [44]:
df_NY = process_state('NEW YORK')
df_NY


NEW YORK party counts:
party
LIBERTARIAN    62
REPUBLICAN     62
DEMOCRAT       62
OTHER          62
Name: count, dtype: int64


,year,state,state_po,totalvotes,votes_republican,votes_democrat,votes_libertarian,votes_other
0,2024,NEW YORK,NY,8262495,3578899,4619195,5338,59063


In [45]:
df_ND = process_state('NORTH DAKOTA')
df_ND


NORTH DAKOTA party counts:
party
LIBERTARIAN    53
REPUBLICAN     53
DEMOCRAT       53
OTHER          53
Name: count, dtype: int64


,year,state,state_po,totalvotes,votes_republican,votes_democrat,votes_libertarian,votes_other
0,2024,NORTH DAKOTA,ND,368155,246505,112327,6227,3096


## OH, OR, TN, VA

In [46]:
df_OH = process_state('OHIO')
df_OH


OHIO party counts:
party
LIBERTARIAN    88
REPUBLICAN     88
DEMOCRAT       88
OTHER          88
Name: count, dtype: int64


,year,state,state_po,totalvotes,votes_republican,votes_democrat,votes_libertarian,votes_other
0,2024,OHIO,OH,5767788,3180116,2533699,28200,25773


In [47]:
df_OR = process_state('OREGON')
df_OR


OREGON party counts:
party
LIBERTARIAN    36
REPUBLICAN     36
DEMOCRAT       36
OTHER          36
Name: count, dtype: int64


,year,state,state_po,totalvotes,votes_republican,votes_democrat,votes_libertarian,votes_other
0,2024,OREGON,OR,2244493,919480,1240600,9061,75352


In [48]:
df_TN = process_state('TENNESSEE')
df_TN


TENNESSEE party counts:
party
REPUBLICAN    95
DEMOCRAT      95
OTHER         95
Name: count, dtype: int64


,year,state,state_po,totalvotes,votes_republican,votes_democrat,votes_libertarian,votes_other
0,2024,TENNESSEE,TN,3063942,1966865,1056265,0,40812


In [49]:
df_VA = process_state('VIRGINIA')
df_VA


VIRGINIA party counts:
party
LIBERTARIAN    133
REPUBLICAN     133
DEMOCRAT       133
OTHER          133
Name: count, dtype: int64


,year,state,state_po,totalvotes,votes_republican,votes_democrat,votes_libertarian,votes_other
0,2024,VIRGINIA,VA,4482576,2075085,2335395,19814,52282


# TOTAL VOTES + other modes

## AZ, AR, IA

In [50]:
df_AZ = process_state5('ARIZONA')
df_AZ


ARIZONA party counts:
party
LIBERTARIAN    15
REPUBLICAN     15
DEMOCRAT       15
OTHER          15
Name: count, dtype: int64


,year,state,state_po,totalvotes,votes_republican,votes_democrat,votes_libertarian,votes_other
0,2024,ARIZONA,AZ,3400986,1770242,1582860,17898,29986


In [51]:
df_AR = process_state6('ARKANSAS')
df_AR


ARKANSAS party counts:
party
LIBERTARIAN    75
REPUBLICAN     75
DEMOCRAT       75
OTHER          75
Name: count, dtype: int64


,year,state,state_po,totalvotes,votes_republican,votes_democrat,votes_libertarian,votes_other
0,2024,ARKANSAS,AR,1182676,759241,396905,5715,20815


In [52]:
df_IA = process_state5('IOWA')
df_IA


IOWA party counts:
party
LIBERTARIAN    99
REPUBLICAN     99
DEMOCRAT       99
OTHER          99
Name: count, dtype: int64


,year,state,state_po,totalvotes,votes_republican,votes_democrat,votes_libertarian,votes_other
0,2024,IOWA,IA,1663506,927019,707278,7218,21991


## LA, OK, PA

In [53]:
df_LA = process_state6('LOUISIANA')
df_LA


LOUISIANA party counts:
party
LIBERTARIAN    64
REPUBLICAN     64
DEMOCRAT       64
OTHER          64
Name: count, dtype: int64


,year,state,state_po,totalvotes,votes_republican,votes_democrat,votes_libertarian,votes_other
0,2024,LOUISIANA,LA,2006975,1208505,766870,6835,24765


In [54]:
df_OK = process_state6('OKLAHOMA')
df_OK


OKLAHOMA party counts:
party
LIBERTARIAN    77
REPUBLICAN     77
DEMOCRAT       77
OTHER          77
Name: count, dtype: int64


,year,state,state_po,totalvotes,votes_republican,votes_democrat,votes_libertarian,votes_other
0,2024,OKLAHOMA,OK,1566173,1036213,499599,9198,21163


In [55]:
df_PA = process_state6('PENNSYLVANIA')
df_PA


PENNSYLVANIA party counts:
party
LIBERTARIAN    67
REPUBLICAN     67
DEMOCRAT       67
OTHER          67
Name: count, dtype: int64


,year,state,state_po,totalvotes,votes_republican,votes_democrat,votes_libertarian,votes_other
0,2024,PENNSYLVANIA,PA,7034206,3543308,3423042,33318,34538


## SC, SD, TX

In [56]:
df_SC = process_state7('SOUTH CAROLINA')
df_SC


SOUTH CAROLINA party counts:
party
LIBERTARIAN    46
REPUBLICAN     46
DEMOCRAT       46
OTHER          46
Name: count, dtype: int64


,year,state,state_po,totalvotes,votes_republican,votes_democrat,votes_libertarian,votes_other
0,2024,SOUTH CAROLINA,SC,2548140,1483747,1028452,12669,23272


In [57]:
df_SD = process_state8('SOUTH DAKOTA')
df_SD


SOUTH DAKOTA party counts:
party
LIBERTARIAN    66
REPUBLICAN     66
DEMOCRAT       66
OTHER          66
Name: count, dtype: int64


,year,state,state_po,totalvotes,votes_republican,votes_democrat,votes_libertarian,votes_other
0,2024,SOUTH DAKOTA,SD,428922,272081,146859,2778,7204


In [58]:
df_TX = process_state6('TEXAS')
df_TX


TEXAS party counts:
party
LIBERTARIAN    254
REPUBLICAN     254
DEMOCRAT       254
OTHER          254
Name: count, dtype: int64


,year,state,state_po,totalvotes,votes_republican,votes_democrat,votes_libertarian,votes_other
0,2024,TEXAS,TX,11388674,6393597,4835250,68557,91270


# nan only

## ID, KS, NC

In [59]:
df_ID = process_state('IDAHO')
df_ID


IDAHO party counts:
party
LIBERTARIAN    44
REPUBLICAN     44
DEMOCRAT       44
OTHER          44
Name: count, dtype: int64


,year,state,state_po,totalvotes,votes_republican,votes_democrat,votes_libertarian,votes_other
0,2024,IDAHO,ID,905057,605246,274972,4462,20377


In [60]:
df_KS = process_state('KANSAS')
df_KS


KANSAS party counts:
party
LIBERTARIAN    105
REPUBLICAN     105
DEMOCRAT       105
OTHER          105
Name: count, dtype: int64


,year,state,state_po,totalvotes,votes_republican,votes_democrat,votes_libertarian,votes_other
0,2024,KANSAS,KS,1327591,758802,544853,7614,16322


In [61]:
df_NC = process_state9('NORTH CAROLINA')
df_NC


NORTH CAROLINA party counts:
party
LIBERTARIAN    400
REPUBLICAN     400
DEMOCRAT       400
OTHER          100
Name: count, dtype: int64

Row counts by county:
13    100
Name: count, dtype: int64


,year,state,state_po,totalvotes,votes_republican,votes_democrat,votes_libertarian,votes_other
0,2024,NORTH CAROLINA,NC,5699141,2898423,2715375,22125,63218


## RI, UT, VT

In [62]:
df_RI = process_state('RHODE ISLAND')
df_RI


RHODE ISLAND party counts:
party
LIBERTARIAN    39
REPUBLICAN     39
DEMOCRAT       39
OTHER          39
Name: count, dtype: int64


,year,state,state_po,totalvotes,votes_republican,votes_democrat,votes_libertarian,votes_other
0,2024,RHODE ISLAND,RI,511816,214291,283750,1614,12161


In [63]:
df_UT = process_state('UTAH')
df_UT


UTAH party counts:
party
LIBERTARIAN    29
REPUBLICAN     29
DEMOCRAT       29
OTHER          29
Name: count, dtype: int64


,year,state,state_po,totalvotes,votes_republican,votes_democrat,votes_libertarian,votes_other
0,2024,UTAH,UT,1488494,883818,562566,16902,25208


In [64]:
df_VT = process_state10('VERMONT')
df_VT


VERMONT party counts:
party
LIBERTARIAN    14
REPUBLICAN     14
DEMOCRAT       14
OTHER          14
Name: count, dtype: int64


,year,state,state_po,totalvotes,votes_republican,votes_democrat,votes_libertarian,votes_other
0,2024,VERMONT,VT,331886,108055,210938,1653,11240


## WA, WV, WI, WY

In [65]:
df_WA = process_state('WASHINGTON')
df_WA


WASHINGTON party counts:
party
LIBERTARIAN    39
REPUBLICAN     39
DEMOCRAT       39
OTHER          39
Name: count, dtype: int64


,year,state,state_po,totalvotes,votes_republican,votes_democrat,votes_libertarian,votes_other
0,2024,WASHINGTON,WA,3924243,1530923,2245849,16428,131043


In [66]:
df_WV = process_state11('WEST VIRGINIA')
df_WV


WEST VIRGINIA party counts:
party
LIBERTARIAN    55
REPUBLICAN     55
DEMOCRAT       55
OTHER          55
Name: count, dtype: int64


,year,state,state_po,totalvotes,votes_republican,votes_democrat,votes_libertarian,votes_other
0,2024,WEST VIRGINIA,WV,762390,533556,214309,3047,11478


In [67]:
df_WI = process_state11('WISCONSIN')
df_WI


WISCONSIN party counts:
party
LIBERTARIAN    72
REPUBLICAN     72
DEMOCRAT       72
OTHER          72
Name: count, dtype: int64


,year,state,state_po,totalvotes,votes_republican,votes_democrat,votes_libertarian,votes_other
0,2024,WISCONSIN,WI,3422918,1697626,1668229,10511,46552


In [68]:
df_WY = process_state12('WYOMING')
df_WY


WYOMING party counts:
party
LIBERTARIAN    23
REPUBLICAN     23
DEMOCRAT       23
OTHER          23
Name: count, dtype: int64


,year,state,state_po,totalvotes,votes_republican,votes_democrat,votes_libertarian,votes_other
0,2024,WYOMING,WY,269048,192633,69527,4193,2695


# other modes, no TOTAL VOTES

In [69]:
df_NM = process_state9('NEW MEXICO')
df_NM


NEW MEXICO party counts:
party
LIBERTARIAN    99
REPUBLICAN     99
DEMOCRAT       99
OTHER          99
Name: count, dtype: int64

Row counts by county:
12    33
Name: count, dtype: int64


,year,state,state_po,totalvotes,votes_republican,votes_democrat,votes_libertarian,votes_other
0,2024,NEW MEXICO,NM,920922,423391,478802,3699,15030


# temp1.1

## CSV

In [70]:
df_combined = pd.concat([
    df_AL, df_AK, df_AZ, df_AR, df_CA, df_CO, df_CT, df_DE, df_DC, df_FL,
    df_GA, df_HI, df_ID, df_IL, df_IN, df_IA, df_KS, df_KY, df_LA, df_ME,
    df_MD, df_MA, df_MI, df_MN, df_MS, df_MO, df_MT, df_NE, df_NV, df_NH,
    df_NJ, df_NM, df_NY, df_NC, df_ND, df_OH, df_OK, df_OR, df_PA, df_RI,
    df_SC, df_SD, df_TN, df_TX, df_UT, df_VT, df_VA, df_WA, df_WV, df_WI, df_WY
], ignore_index=True)

df_combined

,year,state,state_po,totalvotes,votes_republican,votes_democrat,votes_libertarian,votes_other
0,2024,ALABAMA,AL,2264972,1462616,772412,4930,25014
1,2024,ALASKA,AK,338177,184458,140026,3040,10653
2,2024,ARIZONA,AZ,3400986,1770242,1582860,17898,29986
3,2024,ARKANSAS,AR,1182676,759241,396905,5715,20815
4,2024,CALIFORNIA,CA,15862678,6081697,9276179,66662,438140
5,2024,COLORADO,CO,3190873,1377441,1728159,21439,63834
6,2024,CONNECTICUT,CT,1759010,736918,992053,6729,23310
7,2024,DELAWARE,DE,511697,214351,289758,2038,5550
8,2024,DISTRICT OF COLUMBIA,DC,325879,21076,294185,0,10618
9,2024,FLORIDA,FL,10893752,6110125,4683038,31972,68617


In [71]:
df_combined.to_csv("temp1_1_president_2024.csv", index=False, encoding="utf-8-sig")

# temp2

## Functions

In [72]:
def add_vote_rankings(df):
    """
    정당별 득표수에 대해 크기 순위를 매기는 함수

    Parameters:
    -----------
    df : pandas.DataFrame
        votes_republican, votes_democrat, votes_libertarian, votes_other 컬럼을 포함한 데이터프레임

    Returns:
    --------
    pandas.DataFrame
        기존 컬럼 + 순위 컬럼이 포함된 데이터프레임
    """

    # 득표수 컬럼들
    vote_columns = ['votes_republican', 'votes_democrat', 'votes_libertarian', 'votes_other']

    # 각 행에 대해 순위 매기기 (큰 값이 1, 작은 값이 4)
    rankings = df[vote_columns].rank(axis=1, method='min', ascending=False).astype(int)

    # 새로운 컬럼명 지정
    rankings.columns = ['rank_republican', 'rank_democrat', 'rank_libertarian', 'rank_other']

    # 원본 데이터프레임에 순위 컬럼 추가
    result = pd.concat([df, rankings], axis=1)

    return result

# 사용 예시
# result_with_rankings = add_vote_rankings(df)

In [73]:
def create_ranking_summary(df):
    """
    득표수 데이터에서 1위, 2위 정당과 득표율을 계산하는 함수

    Parameters:
    -----------
    df : pandas.DataFrame
        year, state, state_po, totalvotes, votes_republican, votes_democrat,
        votes_libertarian, votes_other 컬럼을 포함한 데이터프레임

    Returns:
    --------
    pandas.DataFrame
        year, state, state_po, first_place_party, first_place_vote_share,
        second_place_party, second_place_vote_share 컬럼을 포함한 데이터프레임
    """

    # 각 행마다 정당별 득표수를 딕셔너리로 만들어 정렬
    def get_top_parties(row):
        party_votes = {
            'REPUBLICAN': row['votes_republican'],
            'DEMOCRAT': row['votes_democrat'],
            'LIBERTARIAN': row['votes_libertarian'],
            'OTHER': row['votes_other']
        }
        sorted_parties = sorted(party_votes.items(), key=lambda x: x[1], reverse=True)
        return pd.Series({
            'first_place_party': sorted_parties[0][0],
            'first_place_votes': sorted_parties[0][1],
            'second_place_party': sorted_parties[1][0],
            'second_place_votes': sorted_parties[1][1]
        })

    result = (df
        .join(df.apply(get_top_parties, axis=1))
        .assign(
            first_place_vote_share=lambda x: round(x['first_place_votes'] / x['totalvotes'], 4),
            second_place_vote_share=lambda x: round(x['second_place_votes'] / x['totalvotes'], 4)
        )
        [['year', 'state', 'state_po', 'first_place_party', 'first_place_vote_share',
          'second_place_party', 'second_place_vote_share']]
    )

    return result

# 사용 예시
# ranking_summary = create_ranking_summary(df)

In [74]:
add_vote_rankings(df_combined)

,year,state,state_po,totalvotes,votes_republican,votes_democrat,votes_libertarian,votes_other,rank_republican,rank_democrat,rank_libertarian,rank_other
0,2024,ALABAMA,AL,2264972,1462616,772412,4930,25014,1,2,4,3
1,2024,ALASKA,AK,338177,184458,140026,3040,10653,1,2,4,3
2,2024,ARIZONA,AZ,3400986,1770242,1582860,17898,29986,1,2,4,3
3,2024,ARKANSAS,AR,1182676,759241,396905,5715,20815,1,2,4,3
4,2024,CALIFORNIA,CA,15862678,6081697,9276179,66662,438140,2,1,4,3
5,2024,COLORADO,CO,3190873,1377441,1728159,21439,63834,2,1,4,3
6,2024,CONNECTICUT,CT,1759010,736918,992053,6729,23310,2,1,4,3
7,2024,DELAWARE,DE,511697,214351,289758,2038,5550,2,1,4,3
8,2024,DISTRICT OF COLUMBIA,DC,325879,21076,294185,0,10618,2,1,4,3
9,2024,FLORIDA,FL,10893752,6110125,4683038,31972,68617,1,2,4,3


In [75]:
df_temp2 = create_ranking_summary(df_combined)
df_temp2

,year,state,state_po,first_place_party,first_place_vote_share,second_place_party,second_place_vote_share
0,2024,ALABAMA,AL,REPUBLICAN,0.6458,DEMOCRAT,0.3410
1,2024,ALASKA,AK,REPUBLICAN,0.5454,DEMOCRAT,0.4141
2,2024,ARIZONA,AZ,REPUBLICAN,0.5205,DEMOCRAT,0.4654
3,2024,ARKANSAS,AR,REPUBLICAN,0.6420,DEMOCRAT,0.3356
4,2024,CALIFORNIA,CA,DEMOCRAT,0.5848,REPUBLICAN,0.3834
5,2024,COLORADO,CO,DEMOCRAT,0.5416,REPUBLICAN,0.4317
6,2024,CONNECTICUT,CT,DEMOCRAT,0.5640,REPUBLICAN,0.4189
7,2024,DELAWARE,DE,DEMOCRAT,0.5663,REPUBLICAN,0.4189
8,2024,DISTRICT OF COLUMBIA,DC,DEMOCRAT,0.9027,REPUBLICAN,0.0647
9,2024,FLORIDA,FL,REPUBLICAN,0.5609,DEMOCRAT,0.4299


In [76]:
df_temp2.to_csv("temp2_president_2024.csv", index=False, encoding="utf-8-sig")

# Batch CSV Files to ZIP

In [77]:
import zipfile
import glob

# Find all CSV files in current directory
csv_files = glob.glob('*.csv')

# Create ZIP file
with zipfile.ZipFile('all_csv_files.zip', 'w') as zipf:
   for file in csv_files:
       zipf.write(file)
       print(f"Added: {file}")  # Show progress

print(f"Total {len(csv_files)} files compressed.")

Added: temp1_1_president_2024.csv
Added: temp2_president_2024.csv
Total 2 files compressed.
